# Context Window Overflow — Memory Pointer Pattern

Based on: [Solving Context Window Overflow in AI Agents](https://arxiv.org/html/2511.22729v1) — IBM Research, 2024

## The Problem

When an AI agent calls a tool that returns a large dataset — logs, database results, documents — that data enters the LLM context window directly as a tool result. For datasets above ~50KB:

- **Cost**: Every subsequent model call re-sends all accumulated data as input tokens
- **Degraded quality**: The LLM is forced to reason over thousands of tokens of raw JSON instead of a clean summary
- **Failure**: For very large datasets, the context window fills completely

The constraint: logs and raw datasets are **indivisible**. Truncating loses critical events. Accurate analysis requires the full data.

## The Solution: Memory Pointer Pattern

Instead of returning raw data to the LLM, the tool:
1. Stores the dataset in `agent.state` — a key-value store on the agent instance, outside LLM context
2. Returns only a pointer string — e.g., `"logs-api-gateway"`
3. Downstream tools read the full data from `agent.state` using that pointer

The LLM sees `"logs-api-gateway"` (~5 tokens). The dataset never enters the context window.

![Without Memory Pointer vs Memory Pointer Pattern](../images/Without-Memory-Pointer.png)

## The Tools

| Tool | Without Pointer | With Pointer |
|------|-----------------|--------------|
| `naive_fetch_logs(app_name, hours)` | Returns full raw JSON — enters LLM context | — |
| `fetch_application_logs(app_name, hours)` | — | Stores logs in `agent.state`, returns pointer key |
| `analyze_error_patterns(logs_pointer)` | — | Reads from `agent.state`, returns error summary JSON |
| `detect_latency_anomalies(logs_pointer)` | — | Reads from `agent.state`, returns latency JSON |
| `generate_incident_report(errors, latency)` | — | Combines analyses into final report |

## What We Test

| Test | What it shows |
|------|---------------|
| 1 — Without Memory Pointer | Naive agent: raw JSON enters LLM context → high token count |
| 2 — With Memory Pointer Pattern | Pointer agent: only key string in context → low token count |
| 3 — Multi-turn reuse | Data fetched once on Turn 1, reused on Turns 2 and 3 |

## Setup: install dependencies

Run the cell below once to install everything from `requirements.txt`. This demo needs **strands-agents 1.44.0+**. Restart the kernel afterward if prompted.

In [ ]:
%pip install -r requirements.txt

import importlib.metadata as _m
_v = _m.version("strands-agents")
assert tuple(int(x) for x in _v.split(".")[:2]) >= (1, 44), (
    f"strands-agents {_v} is too old. This demo needs >= 1.44.0. "
    "Re-run the install cell and restart the kernel."
)
print(f"strands-agents {_v} — OK")

## Configure API Key

Set your OpenAI API key. Get one at https://platform.openai.com/api-keys

> You can swap to any provider supported by Strands — see [Strands Model Providers](https://strandsagents.com/docs/user-guide/concepts/model-providers/) for configuration.

In [ ]:
import os
# os.environ['OPENAI_API_KEY'] = 'your-key-here'  # Uncomment and set your key
assert os.getenv('OPENAI_API_KEY'), (
    '⚠️ OPENAI_API_KEY not set. '
    'Get yours at https://platform.openai.com/api-keys and set it above or in a .env file.'
)

## Setup

In [ ]:
import json, time, secrets, os
from datetime import datetime, timedelta

os.environ['OTEL_SDK_DISABLED'] = 'true'
import logging, warnings  # silence OpenTelemetry 'Failed to detach context' noise
logging.getLogger('opentelemetry').setLevel(logging.CRITICAL)
warnings.filterwarnings('ignore', message='Failed to detach context')

from dotenv import load_dotenv
from strands import Agent, tool, ToolContext
# Using OpenAI-compatible interface via Strands SDK (not direct OpenAI usage)
from strands.models.openai import OpenAIModel
from strands.agent.conversation_manager import SlidingWindowConversationManager
from tools import fetch_application_logs, analyze_error_patterns, detect_latency_anomalies, generate_incident_report

load_dotenv()

MODEL = OpenAIModel(model_id="gpt-4o-mini")

# ─────────────────────────────────────────────────────────────────────────────
# How to switch the model provider (token counting works the same on all of them).
#
# Amazon Bedrock — uses boto3, NO extra package needed.
#   Requires configured AWS credentials (e.g. `aws configure` or environment variables)
#   and model access enabled in the Amazon Bedrock console.
#       from strands.models import BedrockModel
#       MODEL = BedrockModel(
#           model_id="us.anthropic.claude-sonnet-4-20250514-v1:0",
#           region_name="us-east-1",
#       )
#   Docs: https://strandsagents.com/docs/user-guide/concepts/model-providers/amazon-bedrock/
#
# Anthropic (direct API) — requires the extra:  pip install 'strands-agents[anthropic]'
#   The API key goes inside client_args (get one at https://console.anthropic.com/).
#       from strands.models.anthropic import AnthropicModel
#       MODEL = AnthropicModel(
#           client_args={"api_key": os.getenv("ANTHROPIC_API_KEY")},
#           model_id="claude-sonnet-4-6",
#           max_tokens=1028,
#       )
#   Docs: https://strandsagents.com/docs/user-guide/concepts/model-providers/anthropic/
# ─────────────────────────────────────────────────────────────────────────────

# Same query used in both Test 1 and Test 2 for a fair comparison
QUERY = "Fetch 2 hours of logs for 'api-gateway' and analyze error patterns. How many errors occurred and which services had the most?"


# ── Naive tool (WITHOUT memory pointer) ──────────────────────────────────────
@tool
def naive_fetch_logs(app_name: str, hours: int = 2) -> str:
    """Fetch application logs. Returns full raw JSON — no memory pointer pattern."""
    logs = []
    base = datetime.now() - timedelta(hours=hours)
    for i in range(hours * 100):
        logs.append({
            "timestamp": (base + timedelta(seconds=i)).isoformat(),
            "level": ["INFO", "WARN", "ERROR", "DEBUG"][secrets.randbelow(4)],
            "service": ["api-gateway", "auth-service", "db-connector", "cache-layer"][secrets.randbelow(4)],
            "message": f"Event {i}",
            "duration_ms": secrets.randbelow(4991) + 10,
            "status_code": [200, 201, 400, 404, 500, 503][secrets.randbelow(6)],
        })
    return json.dumps(logs)  # ← Full raw JSON enters LLM context directly


# ── Helper: count tokens actually present in conversation ────────────────────
def count_context_tokens(agent) -> int:
    """Estimate tokens from all messages in conversation history."""
    total = 0
    for msg in agent.messages:
        content = msg.get("content", [])
        if isinstance(content, str):
            total += len(content) // 4
        elif isinstance(content, list):
            for block in content:
                if isinstance(block, dict):
                    if "text" in block:
                        total += len(block["text"]) // 4
                    elif "toolResult" in block:
                        for item in block["toolResult"].get("content", []):
                            if "text" in item:
                                total += len(item["text"]) // 4
                    elif "toolUse" in block:
                        total += len(json.dumps(block["toolUse"].get("input", {}))) // 4
    return total


print("✅ Setup complete!")

---
## Test 1 — Without Memory Pointer (Naive Agent)

`naive_fetch_logs` returns the full raw JSON directly. Every log event, every field, as a tool result string. The LLM receives this as input on its next call — the entire dataset enters the context window.

**Query:** *"Fetch 2 hours of logs for 'api-gateway' and analyze error patterns. How many errors occurred and which services had the most?"*

2 hours × 100 events/hour = 200 events. Each event ~350 characters = **~70KB of raw JSON entering LLM context**.

In [ ]:
agent_naive = Agent(
    model=MODEL,
    tools=[naive_fetch_logs],
)

start = time.time()
response_naive = agent_naive(QUERY)
time_naive = time.time() - start
tokens_naive = count_context_tokens(agent_naive)

print(f"⏱️  {time_naive:.1f}s")
print(f"📊 Tokens in context (conversation history): {tokens_naive:,}")
print(f"🔧 Tool calls: {sum(1 for msg in agent_naive.messages for b in msg.get('content', []) if isinstance(b, dict) and 'toolUse' in b)}")

---
## Test 2 — With Memory Pointer Pattern

`fetch_application_logs` stores the dataset in `agent.state` and returns only the pointer key. `analyze_error_patterns` reads from `agent.state` using that key — the raw data never enters the LLM context.

### SlidingWindowConversationManager

In multi-step workflows the agent accumulates messages: tool calls, tool results, model responses. Without management, this history grows unbounded and eventually overflows the context window.

`SlidingWindowConversationManager(window_size=40)` keeps only the last `window_size` messages, trimming older ones automatically while preserving paired tool call/result blocks.

> The Memory Pointer Pattern and `SlidingWindowConversationManager` solve different problems. The pointer prevents large **tool outputs** from entering context. The sliding window prevents the **conversation history** from growing too long. For multi-step workflows, use both.

**Same query as Test 1** — the only difference is the tools.

In [ ]:
agent_pointer = Agent(
    model=MODEL,
    conversation_manager=SlidingWindowConversationManager(window_size=40),
    tools=[fetch_application_logs, analyze_error_patterns],
)

start = time.time()
response_pointer = agent_pointer(QUERY)
time_pointer = time.time() - start
tokens_pointer = count_context_tokens(agent_pointer)

data = agent_pointer.state.get("logs-api-gateway")
data_size = len(json.dumps(data)) if data else 0

print(f"⏱️  {time_pointer:.1f}s")
print(f"📊 Tokens in context (conversation history): {tokens_pointer:,}")
print(f"📦 agent.state['logs-api-gateway']: {data_size:,} bytes — never entered LLM context")
print(f"🔧 Tool calls: {sum(1 for msg in agent_pointer.messages for b in msg.get('content', []) if isinstance(b, dict) and 'toolUse' in b)}")

---
## Test 3 — Multi-turn: Data Fetched Once, Reused Across Turns

When a user asks follow-up questions about the same logs, the agent does not re-fetch the data. The pointer stays in `agent.state` across turns — the 460KB dataset is loaded once and referenced for every subsequent question.

| Turn | Query | What happens |
|------|-------|-------------|
| 1 | Fetch logs + analyze errors | Data stored in `agent.state["logs-payment-service"]` |
| 2 | Detect latency anomalies in the same logs | Reads from `agent.state` — no re-fetch |
| 3 | Which service had the most errors? | Reads from `agent.state` — no re-fetch |

In [ ]:
agent_dialog = Agent(
    model=MODEL,
    conversation_manager=SlidingWindowConversationManager(window_size=40),
    tools=[fetch_application_logs, analyze_error_patterns, detect_latency_anomalies],
)

print("👤 Turn 1: Fetch 6 hours of logs for payment-service and analyze errors\n")
agent_dialog("Fetch 6 hours of logs for payment-service and analyze errors")

print("\n👤 Turn 2: Detect latency anomalies in those same logs\n")
agent_dialog("Now detect latency anomalies in those same logs")

print("\n👤 Turn 3: Which service had the most errors?\n")
agent_dialog("Which service had the most errors in those logs?")

data = agent_dialog.state.get("logs-payment-service")
if data:
    print(f"\n📦 'logs-payment-service': {len(json.dumps(data)):,} bytes — fetched once, reused across 3 turns")
print(f"💬 Conversation: {len(agent_dialog.messages)} messages total")
tokens_multiturn = count_context_tokens(agent_dialog)

In [ ]:
print(f"{'Approach':<40} {'Tokens in context':>18} {'Time':>8} {'Data in agent.state':>20}")
print("-" * 90)
print(f"{'Test 1 — Naive (no pointer)':<40} {tokens_naive:>18,} {time_naive:>6.1f}s {'—':>20}")
print(f"{'Test 2 — Memory Pointer Pattern':<40} {tokens_pointer:>18,} {time_pointer:>6.1f}s {data_size:>18,} B")
print()

if tokens_naive > tokens_pointer > 0:
    reduction = (1 - tokens_pointer / tokens_naive) * 100
    ratio = tokens_naive // tokens_pointer
    print(f"→ {reduction:.0f}% fewer tokens in context with Memory Pointer Pattern ({ratio}x)")
    print(f"→ {data_size:,} bytes of data processed accurately — never entered LLM context")

---
## Summary

### When to Use

- ✅ Tool returns > 50KB of data
- ✅ Data is indivisible (logs, datasets, matrices)
- ✅ Multiple tools operate on the same dataset
- ✅ Multi-turn conversations about a large dataset

### Next Steps

1. ➡️ [Demo 02: MCP Timeout](../02-mcp-timeout-demo/) — Handle external APIs that stop responding
2. ➡️ [Demo 03: Reasoning Loops](../03-reasoning-loops-demo/) — Prevent agents from calling the same tool repeatedly

### References

- [IBM Research: Solving Context Window Overflow in AI Agents](https://arxiv.org/html/2511.22729v1)
- [Strands Agent State](https://strandsagents.com/docs/user-guide/concepts/agents/state/)
- [Strands Conversation Management](https://strandsagents.com/docs/user-guide/concepts/agents/conversation-management/)
- [Code Repository](https://github.com/aws-samples/sample-why-agents-fail)